# Индивидуальная работа №1

Молдавский государственный университет <br>
Факультет Математики и Информатики <br>
Группа I2302 <br>
Выполнил студент: Годорожа Оксана <br>
Преподователь: Ungureanu Valeriu <br>

## Анализ данных о фильмах: описательный, инференциальный и визуальный анализ

## I. Введение

### Цель анализа
Целью данной работы является применение методов статистического анализа к набору данных о фильмах для выявления закономерностей, влияющих на коммерческий и критический успех кинопроизводства.

### Статистические задачи
1. Описать распределение числовых переменных: бюджет, сборы, рейтинг, длительность
2. Выявить частоты и пропорции для категориальных переменных: жанр, язык
3. Построить 95% доверительные интервалы для среднего рейтинга и доли англоязычных фильмов
4. Проверить статистические гипотезы о влиянии бюджета, языка и жанра на успех фильма
5. Провести дисперсионный анализ (ANOVA) средних сборов по жанровым группам

### Описание набора данных
| Параметр | Значение |
|---|---|
| Источник | Kaggle — Top Movies Dataset |
| Количество наблюдений | 4 803 фильма |
| Количество переменных | 24 |
| Период | 1916 – 2016 |

**Числовые переменные:** `budget`, `revenue`, `runtime`, `vote_average`, `vote_count`, `popularity`  
**Категориальные переменные:** `genres`, `original_language`, `status`, `director`


In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('movies.csv')

# Подвыборки
df_fin = df[(df['budget'] > 0) & (df['revenue'] > 0)].copy()
df_fin['release_date'] = pd.to_datetime(df_fin['release_date'], errors='coerce')
df_fin['year'] = df_fin['release_date'].dt.year
df_fin['profit'] = df_fin['revenue'] - df_fin['budget']
df_fin['roi'] = (df_fin['profit'] / df_fin['budget']) * 100

df_rated = df[df['vote_average'] > 0].copy()

print(f"✅ Всего фильмов:                    {len(df)}")
print(f"✅ С бюджетом и сборами (>0):        {len(df_fin)}")
print(f"✅ С ненулевым рейтингом:            {len(df_rated)}")
print(f"✅ Переменных:                       {df.shape[1]}")
print(f"✅ Уникальных режиссёров:            {df['director'].nunique()}")
print(f"✅ Уникальных языков:                {df['original_language'].nunique()}")


✅ Всего фильмов:                    4803
✅ С бюджетом и сборами (>0):        3229
✅ С ненулевым рейтингом:            4740
✅ Переменных:                       24
✅ Уникальных режиссёров:            2349
✅ Уникальных языков:                37


## II. Описательная статистика и визуализация данных

### 2.1 Сводная таблица числовых переменных

В таблице ниже представлены основные описательные статистики для числовых переменных датасета.


In [4]:
stats = df[['budget', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity']].describe().round(2)
stats.index = ['N', 'Среднее', 'Ст. откл.', 'Мин', 'Q1 (25%)', 'Медиана (50%)', 'Q3 (75%)', 'Макс']
stats.columns = ['Бюджет ($)', 'Сборы ($)', 'Длит. (мин)', 'Рейтинг', 'Кол-во оценок', 'Популярность']
stats


,Бюджет ($),Сборы ($),Длит. (мин),Рейтинг,Кол-во оценок,Популярность
N,4.803000e+03,4.803000e+03,4801.00,4803.00,4803.00,4803.00
Среднее,2.904504e+07,8.226064e+07,106.88,6.09,690.22,21.49
Ст. откл.,4.072239e+07,1.628571e+08,22.61,1.19,1234.59,31.82
Мин,0.000000e+00,0.000000e+00,0.00,0.00,0.00,0.00
Q1 (25%),7.900000e+05,0.000000e+00,94.00,5.60,54.00,4.67
Медиана (50%),1.500000e+07,1.917000e+07,103.00,6.20,235.00,12.92
Q3 (75%),4.000000e+07,9.291719e+07,118.00,6.80,737.00,28.31
Макс,3.800000e+08,2.787965e+09,338.00,10.00,13752.00,875.58


**Интерпретация:**
- Среднее значение бюджета ($29M) значительно выше медианы ($15M) — сильная правосторонняя асимметрия, обусловленная блокбастерами
- Рейтинг сосредоточен около 6.09 с умеренным разбросом (σ ≈ 1.19)
- Медианная длительность — 106 мин, что соответствует стандарту индустрии
- Популярность имеет экстремальные выбросы (max = 875.6 при медиане 12.9)


In [5]:
# Пропущенные значения
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False).reset_index()
missing.columns = ['Переменная', 'Пропущено']
missing['Доля (%)'] = (missing['Пропущено'] / len(df) * 100).round(2)
print("=== Пропущенные значения ===")
missing


=== Пропущенные значения ===


,Переменная,Пропущено,Доля (%)
0,homepage,3091,64.36
1,tagline,844,17.57
2,keywords,412,8.58
3,cast,43,0.90
4,director,30,0.62
5,genres,28,0.58
6,overview,3,0.06
7,runtime,2,0.04
8,release_date,1,0.02


### 2.2 Распределение рейтингов фильмов

Гистограмма ниже отображает распределение средней оценки зрителей (`vote_average`).  
На графике нанесены линии среднего (оранжевая) и медианы (красная) для визуального сравнения.


In [ ]:


mean_r = df_rated['vote_average'].mean()
med_r = df_rated['vote_average'].median()
std_r = df_rated['vote_average'].std()

fig = px.histogram(
    df_rated, x='vote_average', nbins=40,
    color_discrete_sequence=['#3498DB'],
    title='Распределение рейтингов фильмов (vote_average)<br>'
          '<sup>Большинство фильмов получают оценку 6–7 | Источник: Kaggle Movies Dataset</sup>'
)
fig.add_vline(x=mean_r, line_dash='dash', line_color='orange', line_width=2,
              annotation_text=f'Среднее: {mean_r:.2f}', annotation_position='top right')
fig.add_vline(x=med_r, line_dash='dot', line_color='red', line_width=2,
              annotation_text=f'Медиана: {med_r:.2f}', annotation_position='top left')
fig.update_xaxes(title_text='Рейтинг (0...10)')
fig.update_yaxes(title_text='Количество фильмов')
fig.show()

print(f"Среднее: {mean_r:.3f} | Медиана: {med_r:.3f} | Ст. откл.: {std_r:.3f}")


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed